In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

from wsp_tasha_toolbox.data_model import MicrosimData

In [ ]:
gta = gpd.read_file(
    r"C:\Users\CALB174209\Projects-local\Oshawa\GIS\tts2016_2006zn_shp.zip")

In [ ]:
zones_gta06_sub = gta[~gta["region"].isin([18, 19, 11, 20])]

In [ ]:
durham_pds = {
    17: "Brcok",
    18: "Uxbridge",
    19: "Scugog",
    20: "Pickering",
    21: "Ajax",
    22: "Whitby ",
    23: "Oshawa",
    24: "Clarington",
}

In [ ]:
durham = gpd.read_file(
    r"C:\Users\CALB174209\Projects-local\Oshawa\GIS\DurhamTrafficZone2020_Shapefile\DTZ_2020.shp")

In [ ]:
print(durham.crs)

In [ ]:
print(zones_gta06_sub.crs)

In [ ]:
# add pd information to durham region shape file 
# Create centroid point for each Durham polygon
durham_centroid = durham.copy()
durham_centroid["geometry"] = durham_centroid.geometry.centroid


# Spatial join: find which GTA polygon contains the Durham centroid
durham_with_gta_info = gpd.sjoin(
 durham_centroid,
 zones_gta06_sub,
 how="left",
 predicate="within"
)


# Remove centroid geometry and keep original Durham polygon geometry
durham_with_gta_info = durham_with_gta_info.drop(columns=["geometry", "index_right"], errors="ignore")

durham_with_gta = durham.merge(
 durham_with_gta_info,
 left_index=True,
 right_index=True,
 how="left",
suffixes=("", "_gta")
)

In [ ]:
durham_with_gta['pd'].value_counts()

In [ ]:
durham_with_gta['pd'].isna().sum()

In [ ]:
durham_with_gta[durham_with_gta["pd"].isna()]

In [ ]:
durham_with_gta.loc[durham_with_gta["ModelZone"]==1398 ,"pd"]=22
durham_with_gta.loc[durham_with_gta["ModelZone"]==1398 ,"gta06"]=1153

In [ ]:
durham_with_gta['pd'].value_counts()

In [ ]:
# create a merged shape file

zones_gta06_sub["edited_zone_number"] = zones_gta06_sub["gta06"]
gta_durham_part = zones_gta06_sub[zones_gta06_sub["pd"].between(17, 24)].copy()

gta_durham_centriod = gta_durham_part.copy()
gta_durham_centriod["geometry"] = gta_durham_centriod.geometry.centroid

#spatial join
gta_durham_joined = gpd.sjoin(
 gta_durham_centriod,durham[["ModelZone","geometry"]],
    how="left"
   # predicate="within"
)
zones_gta06_sub.loc[gta_durham_joined.index, "edited_zone_number"] = gta_durham_joined["ModelZone"]

In [ ]:
zones_gta06_sub["edited_zone_number"].value_counts()    

In [ ]:
zones_gta06_sub.loc[zones_gta06_sub["pd"].between(17, 24),["edited_zone_number"]]

In [ ]:
zones_gta06_sub['edited_zone_number'].isna().sum()

In [ ]:
zones_gta06_sub["edited_zone_number"].duplicated().sum()

In [ ]:
zones_gta06_sub[zones_gta06_sub["edited_zone_number"].duplicated()]

In [ ]:
zones_gta06_sub[zones_gta06_sub["region"]==20]

In [ ]:
zones_gta06_sub = zones_gta06_sub.dissolve(
    by="edited_zone_number",
    as_index=False
)

In [ ]:
zones_gta06_sub[zones_gta06_sub["edited_zone_number"].duplicated()]

In [ ]:
zones_gta06_sub["edited_zone_number"] = zones_gta06_sub["edited_zone_number"].astype(str)

In [ ]:
zones_gta06_sub['region'].value_counts()

In [ ]:
zones_gta06_sub.to_file(r"C:\Users\CALB174209\Projects-local\Oshawa\GIS\zones_edited_20260414.shp")